## Example 1 — dask.array: chunked parallel linear algebra

In [ ]:
import dask.array as da

rng = da.random.default_rng(42)
x = rng.random((1000, 1000), chunks=(250, 250))     # lazy, 16 chunks
gram = x @ x.T                                       # blocked matmul -> per-chunk BLAS
diag_mean = da.diagonal(gram).mean()
total = x.sum()

diag_val, total_val = da.compute(diag_mean, total)  # one fused execution
print(f"npartitions={x.npartitions}  diag_mean={diag_val:.4f}  sum={total_val:.1f}")

# Invariants: x ~ U(0,1) over 1e6 elements -> sum ~ 500k. Each Gram diagonal entry is
# sum_k x[i,k]^2 over 1000 terms with E[x^2]=1/3 -> ~333.
assert x.npartitions == 16
assert 495_000 < total_val < 505_000, total_val
assert 320 < diag_val < 346, diag_val
print("OK — dask.array blocked matmul + reduction verified")

## Example 2 — distributed runtime init (LocalCluster + Client)

In [ ]:
import dask.array as da
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(n_workers=1, threads_per_worker=1, processes=False,
                       dashboard_address=None)
client = Client(cluster)
try:
    assert len(client.scheduler_info()["workers"]) == 1
    fut = client.submit(lambda a, b: a * b, 6, 7)     # remote task round-trip
    assert fut.result() == 42
    arr = da.ones((800, 800), chunks=(200, 200))
    cluster_sum = client.compute(arr.sum()).result()  # graph executed on the cluster
    assert cluster_sum == 800 * 800, cluster_sum
    print(f"workers={len(client.scheduler_info()['workers'])}  submit=42  "
          f"cluster_sum={cluster_sum:.0f}")
finally:
    client.close()
    cluster.close()
print("OK — LocalCluster + Client runtime init verified")

## Example 3 — dask.dataframe: partitioned groupby vs pandas

In [ ]:
# dask.dataframe partitions a pandas frame across 8 partitions; the aggregation result
# is checked against pandas ground truth (a data-integrity invariant, not just a print).
import dask.dataframe as dd
import pandas as pd
import numpy as np

rng = np.random.default_rng(0)
pdf = pd.DataFrame({
    "category": rng.choice(list("ABCD"), size=20_000),
    "amount": rng.uniform(10, 1000, size=20_000),
})
ddf = dd.from_pandas(pdf, npartitions=8)

got = ddf.groupby("category")["amount"].agg(["sum", "count"]).compute().sort_index()
exp = pdf.groupby("category")["amount"].agg(["sum", "count"]).sort_index()

assert ddf.npartitions == 8
assert (got["count"] == exp["count"]).all()
assert np.allclose(got["sum"].to_numpy(), exp["sum"].to_numpy())
assert int(got["count"].sum()) == 20_000
print(got)
print("OK — dask.dataframe groupby matches pandas ground truth")

## Example 4 — dask.delayed: custom task graph

In [ ]:
# dask.delayed builds an arbitrary task graph (fan-out load/summarize -> fan-in total)
# and executes it on the default threaded scheduler, checked against an exact value.
# NB: `dask.delayed` the attribute is the curried decorator function (it shadows the
# submodule), so the Delayed *type* must be imported from the dask.delayed module.
from dask import delayed
from dask.delayed import Delayed

@delayed
def load(i):
    return list(range(i * 10, (i + 1) * 10))

@delayed
def summarize(chunk):
    return sum(chunk)

@delayed
def total(parts):
    return sum(parts)

parts = [summarize(load(i)) for i in range(10)]   # 10 parallel branches
graph = total(parts)
result = graph.compute()

assert isinstance(graph, Delayed)
assert result == sum(range(100)) == 4950, result  # 0+1+...+99
print(f"delayed branches={len(parts)}  result={result}")
print("OK — dask.delayed task graph executed correctly")

## Example 5 — dask.bag: semi-structured map / filter / frequencies

In [ ]:
import dask.bag as db
import json

records = [json.dumps({"id": i, "level": "ERROR" if i % 10 == 0 else "INFO"})
           for i in range(1000)]
bag = db.from_sequence(records, npartitions=8)
parsed = bag.map(json.loads)

n_error = parsed.filter(lambda r: r["level"] == "ERROR").count().compute(scheduler="synchronous")
levels = dict(parsed.pluck("level").frequencies().compute(scheduler="synchronous"))

assert bag.npartitions == 8
assert n_error == 100, n_error
assert levels == {"INFO": 900, "ERROR": 100}, levels
print(f"partitions={bag.npartitions}  errors={n_error}  levels={levels}")
print("OK — dask.bag map/filter/frequencies round-trip verified")